# Import các thư viện

In [207]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt

df = pd.read_csv("Pokemon.csv")
df.head()


,#,Name,Type 1,Type 2,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,Bulbasaur,Grass,Poison,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,80,82,83,100,100,80,1,False
3,3,VenusaurMega Venusaur,Grass,Poison,80,100,123,122,120,80,1,False
4,4,Charmander,Fire,NaN,39,52,43,60,50,65,1,False


# 1. Giới thiệu về data

1.1 Xem thông tin dữ liệu

In [208]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   #           800 non-null    int64 
 1   Name        800 non-null    object
 2   Type 1      800 non-null    object
 3   Type 2      414 non-null    object
 4   HP          800 non-null    int64 
 5   Attack      800 non-null    int64 
 6   Defense     800 non-null    int64 
 7   Sp. Atk     800 non-null    int64 
 8   Sp. Def     800 non-null    int64 
 9   Speed       800 non-null    int64 
 10  Generation  800 non-null    int64 
 11  Legendary   800 non-null    bool  
dtypes: bool(1), int64(8), object(3)
memory usage: 69.7+ KB


1.2 Thống kê mô tả của từng biến

In [209]:
df.describe()

,#,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation
count,800.000000,800.000000,800.000000,800.000000,800.000000,800.000000,800.000000,800.00000
mean,362.813750,69.258750,79.001250,73.842500,72.820000,71.902500,68.277500,3.32375
std,208.343798,25.534669,32.457366,31.183501,32.722294,27.828916,29.060474,1.66129
min,1.000000,1.000000,5.000000,5.000000,10.000000,20.000000,5.000000,1.00000
25%,184.750000,50.000000,55.000000,50.000000,49.750000,50.000000,45.000000,2.00000
50%,364.500000,65.000000,75.000000,70.000000,65.000000,70.000000,65.000000,3.00000
75%,539.250000,80.000000,100.000000,90.000000,95.000000,90.000000,90.000000,5.00000
max,721.000000,255.000000,190.000000,230.000000,194.000000,230.000000,180.000000,6.00000


# 2. Xử lý dữ liệu

**2.1 Xử lý dữ liệu rỗng**

Gần như nửa thuộc tính của `Type 2` rỗng vì nhiều pokemon chỉ có 1 Type => Điền giá trị `Blank` cho các ô trống

In [210]:
df.isnull().sum()

#               0
Name            0
Type 1          0
Type 2        386
HP              0
Attack          0
Defense         0
Sp. Atk         0
Sp. Def         0
Speed           0
Generation      0
Legendary       0
dtype: int64

In [211]:
df = df.fillna(value={'Type 2':'Blank'})
df.head()

,#,Name,Type 1,Type 2,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,Bulbasaur,Grass,Poison,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,80,82,83,100,100,80,1,False
3,3,VenusaurMega Venusaur,Grass,Poison,80,100,123,122,120,80,1,False
4,4,Charmander,Fire,Blank,39,52,43,60,50,65,1,False


**2.2 Kiểm tra dữ liệu trùng lặp**

In [212]:
df.duplicated()

0      False
1      False
2      False
3      False
4      False
       ...  
795    False
796    False
797    False
798    False
799    False
Length: 800, dtype: bool

Chỉ có 721 ID duy nhất (79 ID trùng), nên có thể cả mega/primal/forms được tách riêng.

In [213]:
df['#'].duplicated().sum()

np.int64(79)

**2.3 Kiểm tra tính hợp lý**

Đủ 800 giá trị cho `True` và `False`, không lẫn `F` hay `T`

In [214]:
df['Legendary'].value_counts()

Legendary
False    735
True      65
Name: count, dtype: int64

**2.4 Tạo biến mới**


In [215]:
# Tạo biến "Legendary_Label" để phân loại Pokémon thành "Legendary" và "Non-Legendary"
df["Legendary_Label"] = np.where(df["Legendary"], "Legendary", "Non-Legendary")

# Tạo biến "Has_Type_2" để xác định xem Pokémon có Type 2 hay không
df["Has_Type_2"] = df["Type 2"].notna() & df["Type 2"].ne("Blank")

# Tạo biến "Type2_Label" để phân loại Pokémon thành "2 hệ" và "1 hệ"
df["Type2_Label"] = np.where(df["Has_Type_2"], "2 Hệ", "1 Hệ")

# Tạo biến "Total" là tổng của các chỉ số cơ bản
df["Total"] = df["HP"] + df["Attack"] + df["Defense"] + df["Sp. Atk"] + df["Sp. Def"] + df["Speed"]

# Hiển thị các biến mới cùng với các cột gốc để kiểm tra
df[["Name", "Type 1", "Type 2", "Legendary", "Legendary_Label", "Has_Type_2", "Type2_Label", "Total"]].head()

,Name,Type 1,Type 2,Legendary,Legendary_Label,Has_Type_2,Type2_Label,Total
0,Bulbasaur,Grass,Poison,False,Non-Legendary,True,2 Hệ,318
1,Ivysaur,Grass,Poison,False,Non-Legendary,True,2 Hệ,405
2,Venusaur,Grass,Poison,False,Non-Legendary,True,2 Hệ,525
3,VenusaurMega Venusaur,Grass,Poison,False,Non-Legendary,True,2 Hệ,625
4,Charmander,Fire,Blank,False,Non-Legendary,False,1 Hệ,309


# 3. Data Visualization

## 3.1

In [232]:
fig1 = go.Figure()
fig1.update_layout(
    margin=dict(l=0, r=0, t=0, b=0) 
)
fig1.write_image("1.png")
fig1.add_trace(go.Box(
    y=df["HP"],
    name="HP (with Mean)",
    boxmean=True,
    boxpoints="outliers",
    marker_color="#1f77b4",
    line_color="#1f77b4"
))
fig1.add_trace(go.Box(
    y=df["Attack"],
    name="Attack (Mean and SD)",
    boxmean="sd",
    boxpoints="outliers",
    marker_color="#ff7f0e",
    line_color="#ff7f0e"
))
fig1.add_trace(go.Box(
    y=df["Defense"],
    name="Defense (All points)",
    boxpoints="all",
    jitter=0.25,
    pointpos=0,
    marker_color="#2ca02c",
    line_color="#2ca02c"
))
fig1.add_trace(go.Box(
    y=df["Sp. Atk"],
    name="Sp. Atk (Only whiskers)",
    boxpoints=False,
    marker_color="#d62728",
    line_color="#d62728"
))
fig1.add_trace(go.Box(
    y=df["Sp. Def"],
    name="Sp. Def (Suspected outliers)",
    boxpoints="suspectedoutliers",
    marker_color="#0d5ba7",
    line_color="#0d5ba7"
))
fig1.add_trace(go.Box(
    y=df["Speed"],
    name="Speed (Whiskers and outliers)",
    boxpoints="outliers",
    marker_color="#6baed6",
    line_color="#6baed6"
))

fig1.update_layout(
    title="Phân tán và điểm ngoại lai của toàn bộ 6 chỉ số",
    template="plotly_white",
    yaxis_title="Value"
)
fig1.show()


In [217]:
def simple_kde(values, x_grid):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    n = values.size

    if n < 2:
        return np.zeros_like(x_grid, dtype=float)

    bw = 1.06 * values.std(ddof=1) * (n ** (-1 / 5))
    if (not np.isfinite(bw)) or bw <= 0:
        bw = 1.0

    z = (x_grid[:, None] - values[None, :]) / bw
    return np.exp(-0.5 * z ** 2).sum(axis=1) / (n * bw * np.sqrt(2 * np.pi))


In [218]:
hp_values = df["HP"].dropna().to_numpy(dtype=float)
x_hp = np.linspace(hp_values.min(), hp_values.max(), 300)
kde_hp = simple_kde(hp_values, x_hp)

fig2 = go.Figure()
fig2.add_trace(go.Histogram(
    x=hp_values,
    nbinsx=35,
    histnorm="probability density",
    name="HP",
    marker_color="#1f77b4",
    opacity=0.65
))
fig2.add_trace(go.Scatter(
    x=x_hp,
    y=kde_hp,
    mode="lines",
    name="KDE",
    line=dict(color="#1f77b4", width=3)
))
fig2.add_trace(go.Scatter(
    x=hp_values,
    y=np.full(hp_values.shape, -0.0006),
    mode="markers",
    name="Rug",
    marker=dict(symbol="line-ns-open", color="#1f77b4", size=8),
    hoverinfo="skip",
    showlegend=False
))
fig2.update_layout(
    title="Phân phối mật độ của chỉ số HP",
    template="plotly_white",
    yaxis_title="Density",
    bargap=0.02
)
fig2.show()

In [219]:
attack_values = df["Attack"].dropna().to_numpy(dtype=float)
defense_values = df["Defense"].dropna().to_numpy(dtype=float)

x_min = min(attack_values.min(), defense_values.min())
x_max = max(attack_values.max(), defense_values.max())
x_grid = np.linspace(x_min, x_max, 350)

kde_attack = simple_kde(attack_values, x_grid)
kde_defense = simple_kde(defense_values, x_grid)

fig3 = go.Figure()
fig3.add_trace(go.Histogram(
    x=attack_values,
    nbinsx=35,
    histnorm="probability density",
    opacity=0.45,
    name="Attack",
    marker_color="#1f77b4"
))
fig3.add_trace(go.Histogram(
    x=defense_values,
    nbinsx=35,
    histnorm="probability density",
    opacity=0.45,
    name="Defense",
    marker_color="#ff7f0e"
))
fig3.add_trace(go.Scatter(
    x=x_grid,
    y=kde_attack,
    mode="lines",
    name="Attack KDE",
    line=dict(color="#1f77b4", width=2)
))
fig3.add_trace(go.Scatter(
    x=x_grid,
    y=kde_defense,
    mode="lines",
    name="Defense KDE",
    line=dict(color="#ff7f0e", width=2)
))
fig3.add_trace(go.Scatter(
    x=attack_values,
    y=np.full(attack_values.shape, -0.0008),
    mode="markers",
    marker=dict(symbol="line-ns-open", color="#1f77b4", size=7),
    name="Attack Rug",
    hoverinfo="skip",
    showlegend=False
))
fig3.add_trace(go.Scatter(
    x=defense_values,
    y=np.full(defense_values.shape, -0.0016),
    mode="markers",
    marker=dict(symbol="line-ns-open", color="#ff7f0e", size=7),
    name="Defense Rug",
    hoverinfo="skip",
    showlegend=False
))

fig3.update_layout(
    title="Phân phối mật độ của các chỉ số Attack và Defense.",
    template="plotly_white",
    yaxis_title="Density",
    barmode="overlay"
)
fig3.update_yaxes(range=[-0.0022, None], zeroline=False)
fig3.show()


## 3.2

In [220]:
type_counts_tree = df["Type 1"].value_counts().reset_index()
type_counts_tree.columns = ["Type 1", "Count"]

fig4 = px.treemap(
    type_counts_tree,
    path=["Type 1"],
    values="Count",
    color="Count",
    title="Cơ cấu số lượng Pokémon theo Type 1",
    template="plotly_white",
)
fig4.update_traces(textinfo="label+value")
fig4.show()


In [221]:
type_summary = (
    df.groupby("Type 1", as_index=False)
      .agg(
          count=("Name", "count"),
          avg_total=("Total", "mean"),
          avg_speed=("Speed", "mean"),
          legendary_rate=("Legendary", "mean")
      )
)

fig5 = px.scatter(
    type_summary,
    x="count", y="avg_total",
    size="avg_speed", color="legendary_rate",
    hover_name="Type 1",
    title="Mức độ phổ biến và sức mạnh trung bình của từng hệ",
    template="plotly_white"
)
fig5.show()


In [238]:
# Chỉ lấy các hệ có đủ mẫu (>= 15 pokemon) để dễ nhìn
type_counts = df["Type 1"].value_counts()
types_with_enough = type_counts[type_counts >= 15].index.tolist()
faceting_df = df[df["Type 1"].isin(types_with_enough)].copy()

# Sắp xếp hệ theo số lượng giảm dần
type_order_facet = type_counts[type_counts >= 15].sort_values(ascending=False).index.tolist()

fig_facet = px.scatter(
    faceting_df,
    x="Attack",
    y="Defense",
    color="Type 1",
    size="Total",
    size_max=8,
    facet_col="Type 1",
    facet_col_wrap=4,
    category_orders={"Type 1": type_order_facet},
    hover_name="Name",
    hover_data=["HP", "Sp. Atk", "Sp. Def", "Speed", "Generation"],
    title="Phân bố tương quan Attack vs Defense của các hệ Pokemon",
    template="plotly_white"
)

fig_facet.update_xaxes(range=[0, 160])
fig_facet.update_yaxes(range=[0, 160])
fig_facet.update_layout(height=800, showlegend=False)
fig_facet.show()

## 3.3

In [222]:
fig6 = px.histogram(
    df,
    x="Total", color="Legendary_Label",
    barmode="overlay", nbins=25, opacity=0.65,
    title="Phân phối chỉ số Total giữa Legendary và Non-Legendary",
    template="plotly_white"
)
fig6.show()


In [223]:
fig7 = px.box(
    df, x="Legendary_Label", y="Total", points="outliers",
    title="So sánh chỉ số Total giữa Legendary và Non-Legendary",
    color="Legendary_Label",
    template="plotly_white"
)
fig7.show()

In [224]:
gen_by_leg = (
    df.groupby(["Generation", "Legendary_Label"], as_index=False)["Total"]
      .mean()
      .rename(columns={"Total": "avg_total"})
)

fig8 = px.line(
    gen_by_leg.sort_values(["Legendary_Label", "Generation"]),
    x="Generation", y="avg_total", color="Legendary_Label",
    markers=True,
    title="Xu hướng sức mạnh trung bình theo Generation giữa Legendary và Non-Legendary",
    template="plotly_white"
)
fig8.update_xaxes(dtick=1)
fig8.show()

In [225]:
fig9 = px.scatter(
    df,
    x="Attack", y="Defense",
    color="Legendary_Label",
    size="Total",
    size_max=10,
    trendline="ols",
    opacity=0.7,
    hover_name="Name",
    hover_data=["Type 1", "Type 2", "Generation", "Total", "Speed"],
    title="Tương quan giữa Attack và Defense theo nhóm Pokemon",
    template="plotly_white"
)
fig9.show()

## 3.4

In [226]:
common_types = df["Type 1"].value_counts()
common_types = common_types[common_types >= 20].index.tolist()
common_df = df[df["Type 1"].isin(common_types)].copy()
type_order = (
    common_df.groupby("Type 1")["Total"]
    .median()
    .sort_values(ascending=False)
    .index
    .tolist()
)

fig10 = px.box(
    common_df,
    x="Type 1", y="Total",
    category_orders={"Type 1": type_order},
    points="outliers",
    title="Phân bố chỉ số Total theo hệ chính",
    color="Type 1",
    template="plotly_white"
)
fig10.show()


In [227]:
fig11 = px.box(
    df, x="Type2_Label", y="Total", points="outliers",
    title="So sánh chỉ số Total giữa Pokémon 1 hệ và 2 hệ",
    color="Type2_Label",
    template="plotly_white"
)
fig11.show()


In [228]:
fig12 = px.histogram(
    df.sort_values("Generation"),
    x="Total",
    facet_col="Generation",
    facet_col_wrap=3,
    nbins=18,
    title="Phân phối chỉ số Total theo từng Generation",
    color="Generation",
    template="plotly_white"
)
fig12.show()

In [229]:
stats_cols = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]

fig13 = px.scatter_matrix(
    df,
    dimensions=stats_cols,
    color="Generation",
    opacity=0.65,
    title="Ma trận phân tán của các chỉ số cơ bản theo Generation",
    template="plotly_white"
)
fig13.update_traces(diagonal_visible=False, marker=dict(size=4))
fig13.update_layout(height=900, width=900)
fig13.show()


## 3.5

In [230]:
fire_df = df[df["Type 1"] == "Fire"].copy()

fig14 = px.scatter(
    fire_df,
    x="Attack",
    y="Defense",
    size="Total",
    color="Generation",
    hover_name="Name",
    size_max=15,
    title="Chỉ số Attack vs Defense của Pokémon hệ Fire theo Generation",
    template="plotly_white"
)
fig14.update_layout(
    xaxis_range=[0, max(200, fire_df["Attack"].max() + 10)],
    yaxis_range=[0, max(200, fire_df["Defense"].max() + 10)]
)
fig14.show()


In [231]:
fire_pokemon = df[df["Type 1"] == "Fire"].copy()

# Chọn 2 pokemon hệ lửa bất kì
pokemon1 = fire_pokemon.iloc[0]  
pokemon2 = fire_pokemon.iloc[6]  

stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]

stats_closed = stats + [stats[0]]
r1_closed = [pokemon1[stat] for stat in stats] + [pokemon1[stats[0]]]
r2_closed = [pokemon2[stat] for stat in stats] + [pokemon2[stats[0]]]

fig_15 = go.Figure()

fig_15.add_trace(go.Scatterpolar(
    r=r2_closed,
    theta=stats_closed,
    fill='toself',
    mode='lines+markers',
    name=pokemon2['Name'],
    line=dict(color='#ffa500', width=2),
    marker=dict(size=6, color='#ffa500'),
    fillcolor='rgba(255, 165, 0, 0.3)'
))

fig_15.add_trace(go.Scatterpolar(
    r=r1_closed,
    theta=stats_closed,
    fill='toself',
    mode='lines+markers',
    name=pokemon1['Name'],
    line=dict(color='#ff6b6b', width=2),
    marker=dict(size=6, color='#ff6b6b'),
    fillcolor='rgba(255, 107, 107, 0.3)'
))
fig_15.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, max(pokemon1[stats].max(), pokemon2[stats].max()) + 10]
        )
    ),
    title=f"So sánh chỉ số sức mạnh giữa {pokemon1['Name']} vs {pokemon2['Name']}",
    template="plotly_white",
    showlegend=True,
    height=700,
    width=800
)
fig_15.show()